<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/vllm_tpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vLLM on a Colab TPU (v5e-1)

Runtime → Change runtime type → **v5e-1 TPU**.

**Read this first.** One v5e chip has **16 GB of HBM**. `Qwen/Qwen3.8-27B` is about 27.8B parameters, roughly **55 GB in bf16**, so it cannot fit.
vLLM's TPU backend loads safetensors weights (bf16 or supported quantizations), not the GGUF Q4 file used in the llama.cpp notebook. There is no official FP8/AWQ/GPTQ build of this model to shrink it either.
This notebook therefore checks whether the model fits, and if not, **falls back to a smaller Qwen model** so you still get a working vLLM-on-TPU run. Whether the 27B architecture is even supported by the TPU backend is unverified.

## 1. Check the TPU and host

In [ ]:
!free -h | head -2
!df -h /content | tail -1
import os; print({k: v for k, v in os.environ.items() if "TPU" in k or "PJRT" in k})

## 2. Config

In [ ]:
TARGET_MODEL    = "Qwen/Qwen3.8-27B"   # what you asked for
TARGET_PARAMS_B = 27.8                  # billions of parameters
FALLBACK_MODEL  = "Qwen/Qwen3-4B"      # ~8 GB in bf16; fits one v5e chip
FALLBACK_PARAMS_B = 4.0

HBM_GB          = 16
BYTES_PER_PARAM = 2          # bf16
KV_AND_OVERHEAD_GB = 3       # rough headroom for KV cache and activations
MAX_MODEL_LEN   = 4096
FORCE_TARGET    = False      # True = try the 27B anyway (expect an out-of-memory failure)
REFRESH_WHEELS = False   # True = re-download the packages and re-save them on Drive

## 3. Does it fit?

In [ ]:
def need_gb(params_b): return params_b * BYTES_PER_PARAM + KV_AND_OVERHEAD_GB

print(f"{TARGET_MODEL}: needs about {need_gb(TARGET_PARAMS_B):.0f} GB, chip has {HBM_GB} GB")
if need_gb(TARGET_PARAMS_B) <= HBM_GB or FORCE_TARGET:
    MODEL_ID = TARGET_MODEL
else:
    MODEL_ID = FALLBACK_MODEL
    print(f"Does not fit. Falling back to {FALLBACK_MODEL} (about {need_gb(FALLBACK_PARAMS_B):.0f} GB).")
print("Using:", MODEL_ID)

## 4. Keep the Hugging Face cache on Google Drive (models are downloaded once and reused)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
HF_CACHE = "/content/drive/MyDrive/llm_models/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE          # inherited by the vllm server started below
print("Hugging Face cache:", HF_CACHE)

## 5. Install vLLM with the TPU backend (packages are saved on Google Drive after the first install)

In [ ]:
import sys, os, glob, subprocess

WHEELS = f"/content/drive/MyDrive/llm_models/builds/vllm_tpu_py{sys.version_info.major}{sys.version_info.minor}"
PKGS = ["vllm-tpu"]
# Package names and versions for the TPU backend change often; see https://github.com/vllm-project/tpu-inference
# If "vllm-tpu" fails, try PKGS = ["tpu-inference", "vllm"]

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    open("/content/pip_install.log", "a").write(r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError("pip failed:\n" + (r.stdout + r.stderr)[-3000:])

open("/content/pip_install.log", "w").close()
cached = bool(glob.glob(f"{WHEELS}/*.whl")) and not REFRESH_WHEELS
if cached:
    print("Installing from the wheels saved on Drive...")
    try:
        pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
    except RuntimeError as e:
        print("Cached install failed, rebuilding the wheel cache:\n", str(e)[:500])
        cached = False
if not cached:
    print("Downloading packages and saving them to Drive (slow, one time)...")
    os.makedirs(WHEELS, exist_ok=True)
    pip("wheel", "-q", "-w", WHEELS, *PKGS)
    pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
print("Done. Wheels:", len(glob.glob(f"{WHEELS}/*.whl")))

In [ ]:
import importlib.metadata as md_
for pkg in ("vllm", "vllm-tpu", "tpu-inference", "jax", "jaxlib"):
    try: print(pkg, md_.version(pkg))
    except md_.PackageNotFoundError: print(pkg, "not installed")
import jax; print(jax.devices())

## 6. Start the server (the first start compiles with XLA and can take several minutes)

In [ ]:
import subprocess, time, requests, secrets

API_KEY = secrets.token_urlsafe(24)   # required by every request, including through the public tunnel

server = subprocess.Popen(
    ["vllm", "serve", MODEL_ID,
     "--max-model-len", str(MAX_MODEL_LEN),
     "--tensor-parallel-size", "1",
     "--host", "127.0.0.1", "--port", "8000", "--api-key", API_KEY],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(600):                      # up to ~20 min
    if server.poll() is not None:
        print("server exited:\n", open("/content/vllm.log").read()[-4000:]); break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/vllm.log").read()[-4000:])

## 7. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions", headers={"Authorization": f"Bearer {API_KEY}"}, json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens, "temperature": temperature,
    }, timeout=900)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

print(chat("Explain what a TPU is in two sentences."))

## 8. Chat web UI (Gradio)

vLLM has no built-in chat page, so this starts a small Gradio chat that talks to the server. It prints a public `gradio.live` link.
Installing Gradio can change other package versions; if the server was already running, it keeps working, but restart the runtime before reinstalling vLLM.

In [ ]:
%pip install -q gradio
import gradio as gr, json

def stream_chat(message, history):
    msgs = [{"role": m["role"], "content": m["content"]} for m in history] + [{"role": "user", "content": message}]
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"model": MODEL_ID, "messages": msgs, "max_tokens": 1024, "stream": True},
                      stream=True, timeout=900)
    r.raise_for_status()
    text = ""
    for line in r.iter_lines():
        if not line.startswith(b"data: ") or line == b"data: [DONE]":
            continue
        delta = json.loads(line[6:])["choices"][0]["delta"].get("content") or ""
        text += delta
        yield text

demo = gr.ChatInterface(stream_chat, type="messages", title=MODEL_ID)
demo.queue().launch(share=True, prevent_thread_lock=True)   # prints a public gradio.live link

## 9. Use the OpenAI-compatible API from outside Colab

This opens a free Cloudflare tunnel to the vLLM server. The URL is public but every request needs the API key. Anyone with both can use your TPU, so don't share them.

In [ ]:
import subprocess, re, time, os, urllib.request

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)

def open_tunnel(port):
    """Expose a local port through a free Cloudflare quick tunnel and return (process, public URL)."""
    log_path = f"/content/cloudflared_{port}.log"
    proc = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                            stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError("No tunnel URL yet:\n" + open(log_path).read()[-2000:])

In [ ]:
tunnel, PUBLIC_URL = open_tunnel(8000)
print("API    :", PUBLIC_URL + "/v1")
print("API key:", API_KEY)
print("Model  :", MODEL_ID)
print(f"""
from openai import OpenAI
client = OpenAI(base_url="{PUBLIC_URL}/v1", api_key="{API_KEY}")
print(client.chat.completions.create(model="{MODEL_ID}", messages=[{{"role": "user", "content": "Hello"}}]).choices[0].message.content)
""")

## Notes
- Check `/content/vllm.log` for errors. Unsupported architectures, out-of-memory, and version mismatches show up there.
- Host RAM on the Colab TPU runtime is limited (about 12 GB reported), which also constrains loading large checkpoints.
- Models are cached on Google Drive (`MyDrive/llm_models/hf_cache`), so later sessions reuse them. The first load from Drive can be slow.
- The tunnel and API key last only for this session. Run the tunnel cell again for a new URL after a restart.
- The vLLM packages are saved on Drive (`MyDrive/llm_models/builds`) as wheels, so later sessions install without downloading. Set `REFRESH_WHEELS = True` to update them. Install errors are in `/content/pip_install.log`.